In [8]:
import numpy as np
import json
from classy import Class
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from FishLSS.fisherForecast import fisherForecast
from FishLSS.experiment import experiment

In [2]:
# filename for example survey
bfn = 'DESI'
# bfn = 'example_survey'
bd = '/home/adrien/PDM/code/PDM2026_wsl/derivatives/'

## Steps
- define survey (create example_survey.txt with three columns: redshift, $b(z)$, $n(z)$)
- open `example_setup_BAO_sigma8.py` and set `bd`, zmin, zmax, nbins, fsky. Or give nbins and fsky as parameters when runing.

## For DESI
- $n(z)$ can be seen in Fig4 of 2503.14738
- $b(z)$ scaling approximation
- $f_\mathrm{sky}$ --> BGS 12355 $\mathrm{deg}^2$, LRG 10031 $\mathrm{deg}^2$, ELG 10352 $\mathrm{deg}^2$, QSO 11182 $\mathrm{deg}^2$. The full sky : $4 \pi 180^2 / \pi^2 \approx 41253$ $\mathrm{deg}^2$. Most conservative is LRG with $f_\mathrm{sky} \sim 0.243$
- zbins should be given but we'll see that later for now well take the linspaced bins

-------

##  <span style="color:red"> (0) To speed up the process: </span>

It is significantly faster to set up the forecast (and compute derivatives) with a dedicated computing node rather than from within a jupyter notebook. To do so, run the following from an interactive node (from within the `FishLSS` directory). 

Change `bd` in `example_setup_BAO_sigma8.py` before running this. The derivatives will be saved in `bd+'ouput/'+bfn`

Run :
```
python example_setup_BAO_sigma8.py example_survey setup &
python example_setup_BAO_sigma8.py example_survey rec &
python example_setup_BAO_sigma8.py example_survey fs &
python example_setup_BAO_sigma8.py example_survey lens &
wait
```

and replace `example survey` with `bfn`. If any of these process run over the time limit, simply rerun them. $\verb|FishLSS|$ will pick up where it left off.

Optional: the number of bins and $f_\text{sky}$ can be passed as arguments when calling `example_setup_BAO_sigma8.py`, should you want to change them from their default values (3, 0.5 respectively for this example).

-------

python example_setup_BAO_sigma8.py DESI setup 2 0.25 &
python example_setup_BAO_sigma8.py DESI rec 2 0.25 &
python example_setup_BAO_sigma8.py DESI fs 2 0.25 &
python example_setup_BAO_sigma8.py DESI lens 2 0.25 &
wait

## (1) Setting up a $\verb|FishLSS|$ forecast

Most of the work has already been done in step <span style="color:red"> (0) </span>. After setting up a forecast, $\verb|FishLSS|$ will create a JSON file that summarizes the forecast assumptions and inputs. We will use this to load the relevant information when building the forecast from within the notebook.

In [3]:
f = open('../derivatives/output/'+bfn+'/summary.json')
summary = json.load(f)

In [4]:
print('summary = ', summary)

summary =  {'Forecast name': 'DESI', 'Edges of redshift bins': [0.475, 0.9999999999999999, 1.525], 'Centers of redshift bins': [0.7374999999999999, 1.2625], 'Linear Eulerian bias in each bin': [1.2240418698784212, 1.539051579193082], 'Number density in each bin': [0.0006271179724236919, 0.0007563077105658357], 'fsky': 0.25, 'CLASS default parameters': {'output': 'tCl lCl mPk', 'l_max_scalars': 2000, 'lensing': 'yes', 'non linear': 'halofit', 'A_s': 2.10732e-09, 'n_s': 0.96824, 'alpha_s': 0.0, 'h': 0.677, 'N_ur': 1.0196, 'N_ncdm': 2, 'm_ncdm': '0.01,0.05', 'tau_reio': 0.0568, 'omega_b': 0.02247, 'omega_cdm': 0.11923, 'Omega_k': 0.0, 'P_k_max_h/Mpc': 2.0, 'z_pk': '0.0,6'}}


In [6]:
params = summary['CLASS default parameters']
cosmo = Class() 
cosmo.set(params) 
cosmo.compute() 

### (1b) The experiment

Now we specify the experiment, which is an instance of $\verb|experiment.py|$. At a minimum, we need to specify the redshift range of the survey ($z_\text{min}$ and $z_\text{max}$), the redshift binning, the sky coverage $f_\text{sky}$, the linear bias $b(z)$, and the number density $\bar{n}(z)$. The redshift binning can be specified in two ways: you can either input a `zedges` (numpy array) to specify the edges of the bins, or `nbins` (integer), in which case the redshift bins are assumed to be linearly spaced in $z$. 

In [29]:
# load fiducial linear bias/number density from table
zs,bs,ns = np.genfromtxt('/home/adrien/PDM/code/FishLSS/notebooks/' + bfn + '.txt').T

# assume zs spans the full survey
ze = summary['Edges of redshift bins']
zmin = ze[0]
zmax = ze[-1]

z_centers = (np.array(ze[1:])+np.array(ze[:-1]))/2

# interpolate
b = interp1d(zs,bs)
n = interp1d(zs,ns)

nbins = len(ze)-1
fsky = summary['fsky']

We can now create the experiment:

In [10]:
exp = experiment(zmin=zmin, zmax=zmax, nbins=nbins, fsky=fsky, b=b, n=n)

In addition to the above, one can optionally specify the following in an `experiment` object.

- `b2` (function of $z$): quadratic bias $b_2(z)$ of the tracer, default $b_2 = 8(b-1)/21$
- `sigv` (float): the comoving velocity dispersion for FoG contributions [km/s], default is 100 km/s
- `sigma_z` (float): redshift error $\sigma_z/(1+z)$, assumed to be independent of redshift, default is 0

### (1c) The forecast object

With a cosmology and an experiment in hand, we can now create a forecast. Running the line below will create an `output` directory, as well as a subdirectory for the experiment of interest: `output/example_survey` in this case. After creating these directories, $\verb|FishLSS|$ will calculate the fiducial power spectra ($P_{gg}(\boldsymbol{k},z)$ and $P_\text{recon}(\boldsymbol{k},z)$) at the center of each redshift bin, and store them in `output/example_survey/derivatives/` and `output/example_survey/derivatives_recon` respectively (assuming that the files don't already exist). $\verb|FishLSS|$ will also calculate $C^{\kappa\kappa}_\ell$, $C^{\kappa g_i}_\ell$ and $C^{g_i g_i}_\ell$ for each redshift bin, and save them in `output/example_survey/derivatives_Cl`. 

In [11]:
# By default FishLSS won't recompute the fiducial power spectra (unless overwrite=True),
# instead, FishLSS will load them into memory. (assuming "python example_BAO_sigma8.py setup"
# has been run)
name = summary['Forecast name']
forecast = fisherForecast(experiment=exp,cosmo=cosmo,name=name,basedir=bd)

Redshift-space power spectra are computed on a (flattened) $k-\mu$ grid. That is, $P_{gg}(\boldsymbol{k},z)$ is stored as an array of length `forecast.Nk * forecast.Nmu`, with the corresponding values of $k$ and $\mu$ stored in `forecast.k` and `forecast.mu`. 

## (2) BAO forecast

As described in $\S3.6$ of [2106.09713](https://arxiv.org/pdf/2106.09713.pdf), we hold the shape of the fiducial power spectrum fixed in our BAO forecasts. We then find the errors on the two A-P parameters ($\alpha_\perp$, $\alpha_\parallel$) after marginalizing over the linear bias $b$ and 15 broad-band polynomials $\sum_{n=0}^4\sum_{m=0}^2 c_{nm}k^n\mu^{2m}$. We finally intepret the errors on the A-P parameters as the relative errors of $D_A(z)/r_d$ and $H(z)r_d$, where $r_d$ is sound horizon at the drag epoch.

Marginalizing over the polynomial coefficients is trivial to do analytically, so we only need to numerically compute derivatives with respect to $\alpha_\perp,\alpha_\parallel$ and $b$. 

In [12]:
basis = np.array(['alpha_perp','alpha_parallel','b'])

# set recon = True, so that we perform BAO reconstruction when computing the power spectrum
forecast.recon = True

# set the "marginalized parameters", aka the derivatives, to be [alpha's, linear b]
forecast.free_params = basis

In [ ]:
# derivatives should have already been computed using "python example_BAO_sigma8.py example_survey rec"

# can also simply use:
# forecast.compute_derivatives()
# but this is slow in Jupyter

The derivatives have been automatically stored in `output/example_survey/derivatives_recon`. To load these derivatives, simply run:

In [13]:
derivs = forecast.load_derivatives(basis)

Now let's compute the Fisher matrices in each redshift bin using `get_fisher`, which takes the arguments:

- `basis` (np array): basis of the Fisher matrix 
- `globe` (int): let's not worry about this for now, it's value isn't important for computing the Fisher matrix for a single redshift bin
- `derivatives` (np array): load the derivatives from memory, if not specificied $\verb|FishLSS|$ will recalculate them (takes a lot of time!)
- `zbins` (np array): an array of ints specifying which redshift bins to include in the Fisher matrix. In this case we're computing a Fisher matrix for each redshift bin, so we set `zbins = np.array([i])` to get the Fisher matrix for the i'th bin.

In addition the the above you can also specify `kmax` or `kmax_knl` (ratio of $k_\text{max}$ to the non-linear scale at the center of each redshift bin). By default we set `kmax_knl=1` and `kmax=-10`. If `kmax` is set to be a positive number, then the code ignores `kmax_knl` and uses `kmax` instead.

In [15]:
# get the fisher matrices in each of the 3 redshift bins
F = lambda i: forecast.gen_fisher(basis, 100, derivatives=derivs, zbins=np.array([i]))
Fs = [F(i) for i in range(nbins)]
Fs = np.array(Fs)

Since we set `recon = True` when computing the Fisher matrices, $\verb|FishLSS|$ automatically knows to marginalize over the 15 polynomials, so each Fisher matrix will be an $18\times18$ matrix with basis $\{\alpha_\perp,\alpha_\parallel,b,c_{00},\cdots\}$

In [16]:
print(Fs.shape)

(2, 18, 18)


Now lets invert and compute the errors on the A-P parameters.

In [17]:
Finvs = [np.linalg.inv(Fs[i]) for i in range(nbins)]
saperp = [np.sqrt(Finvs[i][0,0]) for i in range(nbins)]
saparr = [np.sqrt(Finvs[i][1,1]) for i in range(nbins)]

From which we get the relative error on $D_A(z)/r_d$ and $H(z)r_d = r_d/D_H(z)$ in each redshift bin:

In [18]:
print('Relative error on DA/rd:',saperp)
print('Relative error on H*rd:',saparr)

Relative error on DA/rd: [np.float64(0.0064000788278571075), np.float64(0.004072793144731653)]
Relative error on H*rd: [np.float64(0.009112377873948), np.float64(0.005961780596589008)]


(For absolute errors : If $\sigma_\mathrm{parr}=$`saparr`, error on $D_H(z)/r_d$ is $\sigma_\mathrm{parr}/\alpha_\mathrm{parr}^2$)

But we have relative errors so it's the same

(Since $D_A(z) = D_M/(1+z)$ then if $a = D_A(z)/r_d$, we have $D_M(z)/r_d = a\cdot (1+z)$ and error : $\sigma_a \cdot (1+z)$)

In fact the rel. error on $D_M(z)/r_d$ is $\sqrt{(\sigma_a/a)^2 + (\sigma_z/(1+z))^2}$ which is then the same if we consider $\sigma_z = 0$

In [34]:
a_perp = np.array(saperp) * (1 + z_centers)
print('Absolute error on DA:',a_perp)

Absolute error on DA: [0.01112014 0.00921469]


We're done with BAO forecasting, so let's set `recon = False`

In [ ]:
forecast.recon = False